# NSE Stocks — Monthly Expiry Move Table (Daily Candles)

One notebook for all F&O stocks in `data/raw/option_selling/stocks`. Pick a symbol in the parameters cell (or from the dropdown at the bottom) and the whole notebook re-reads for it.

For every consecutive pair of **monthly** stock expiries, measured on daily candles from the previous expiry's close:
- the max upside % and max downside % reached anywhere inside the expiry cycle,
- the net close-to-close move settled at the next expiry,
- how often a strike was *touched* intra-cycle but still expired worthless,
- which cycles settled outside a percentile survivability band (Wald-style outlier analysis),
- and a cross-symbol screen ranking every stock by how often it stayed inside a chosen band.

Source data:
- `data/raw/option_selling/stocks/nse_stock_expiry.csv` — scheduled and actual monthly expiry dates
- `data/raw/option_selling/stocks/<symbol>_<start>_<end>.json` — daily OHLC candles per symbol

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    """Walk up from the notebook's location until the folder holding data/raw is found."""
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "option_selling" / "stocks"
EXPIRY_CSV = RAW_DIR / "nse_stock_expiry.csv"

# --- Parameters -------------------------------------------------------------
SYMBOL = "reliance"                                  # any symbol listed in the next cell
SKIP_FIRST_CANDLES = 0                               # drop the first N daily candles after each expiry
STRIKE_DISTANCES = [2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 12.0]   # % away from the prior expiry close
SCREEN_DISTANCE = 5.0                                # band used by the cross-symbol screen
BIN_WIDTH, LOWER_TAIL, UPPER_TAIL = 2.5, -15.0, 15.0            # return-distribution buckets
GAP_FLAG_PCT = 25.0                                  # single-day moves beyond this are flagged as suspect
OUTLIER_COVERAGE = 80                                # % of cycles kept inside the survivability band
# ---------------------------------------------------------------------------

# Diverging poles with a neutral midpoint: upside = warm, downside = cool.
UP_COLOR = (235, 104, 52)
DOWN_COLOR = (42, 120, 214)
NEUTRAL = (240, 239, 236)
UP_HEX, DOWN_HEX = "#eb6834", "#2a78d6"
SERIES_HEX = ["#2a78d6", "#eb6834", "#1baf7a"]   # fixed categorical order, never cycled
HEADER_FILL = "#2b2d42"

pd.options.display.float_format = "{:,.2f}".format

## Available Symbols

File names follow `<symbol>_<start>_<end>.json`, so the symbol is everything before the last two underscores.

In [2]:
SYMBOL_FILES = {path.stem.rsplit("_", 2)[0]: path for path in sorted(RAW_DIR.glob("*.json"))}
SYMBOLS = sorted(SYMBOL_FILES)

if SYMBOL not in SYMBOL_FILES:
    raise KeyError(f"Unknown symbol {SYMBOL!r}. Available: {', '.join(SYMBOLS)}")

print(f"{len(SYMBOLS)} symbols available:")
print(", ".join(SYMBOLS))
print(f"\nSelected: {SYMBOL.upper()} → {SYMBOL_FILES[SYMBOL].name}")

48 symbols available:
adanient, adaniports, apollohosp, asianpaint, axisbank, bajaj-auto, bajajfinsv, bajfinance, bel, bhartiartl, cipla, coalindia, drreddy, eichermot, eternal, grasim, hcltech, hdfcbank, hdfclife, hindalco, hindunilvr, icicibank, indigo, infy, itc, jswsteel, kotakbank, lt, m&m, maruti, maxhealth, nestleind, ntpc, ongc, powergrid, reliance, sbilife, sbin, shriramfin, sunpharma, tataconsum, tatasteel, tcs, techm, titan, trent, ultracemco, wipro

Selected: RELIANCE → reliance_2015-01-01_2026-08-31.json


## Load Expiries & Candles

Each expiry date is **snapped back to the last session that actually traded on or before it**, so a holiday-shifted expiry still anchors on a real closing print; duplicates after snapping are dropped. Symbols that listed part-way through the calendar simply get fewer cycles.

Candles are used exactly as they appear in the raw dump. If a symbol's series is not corporate-action adjusted, a split, bonus or demerger shows up as a single-day collapse — the loader prints any day beyond ±`GAP_FLAG_PCT` so those cycles can be discounted by eye rather than silently trusted.

In [3]:
expiry_df = pd.read_csv(EXPIRY_CSV, parse_dates=["Actual_Expiry_Date"])
EXPIRY_DATES = expiry_df["Actual_Expiry_Date"].dt.normalize().sort_values().reset_index(drop=True)


def load_candles(symbol: str) -> pd.DataFrame:
    """Load one symbol's daily OHLC candles, indexed by session date."""
    with open(SYMBOL_FILES[symbol]) as f:
        raw = json.load(f)

    df = pd.DataFrame(
        raw["data"]["candles"],
        columns=["timestamp", "open", "high", "low", "close", "volume", "oi"],
    )
    df["timestamp"] = (
        pd.to_datetime(df["timestamp"], utc=True)
        .dt.tz_convert("Asia/Kolkata")
        .dt.tz_localize(None)
        .dt.normalize()
    )
    df = df.drop_duplicates("timestamp").set_index("timestamp").sort_index()
    return df[["open", "high", "low", "close", "volume"]].apply(pd.to_numeric)


def expiry_sessions_for(price: pd.DataFrame) -> pd.Series:
    """Snap every expiry inside the price history back to the last session that traded."""
    sessions = price.index
    in_range = EXPIRY_DATES[(EXPIRY_DATES >= sessions.min()) & (EXPIRY_DATES <= sessions.max())]
    snapped = pd.Series(sessions[sessions.searchsorted(in_range, side="right") - 1])
    return snapped.drop_duplicates().reset_index(drop=True)


def build_windows(price: pd.DataFrame, skip_first: int = 0) -> pd.DataFrame:
    """Measure every expiry-to-expiry cycle, anchored on the prior expiry's daily close."""
    expiries = expiry_sessions_for(price)
    rows = []
    for prev_expiry, curr_expiry in zip(expiries.iloc[:-1], expiries.iloc[1:]):
        window = price.loc[(price.index > prev_expiry) & (price.index <= curr_expiry)]
        if skip_first:
            window = window.iloc[skip_first:]
        if window.empty:
            continue

        base_close = price.at[prev_expiry, "close"]
        up_date, down_date = window["high"].idxmax(), window["low"].idxmin()
        expiry_close = window["close"].iloc[-1]

        rows.append(
            {
                "prev_expiry": prev_expiry,
                "curr_expiry": curr_expiry,
                "base_close": base_close,
                "max_up_pct": (window.at[up_date, "high"] - base_close) / base_close * 100,
                "max_up_date": up_date,
                "max_down_pct": (window.at[down_date, "low"] - base_close) / base_close * 100,
                "max_down_date": down_date,
                "expiry_close": expiry_close,
                "expiry_move_pct": (expiry_close - base_close) / base_close * 100,
                "trading_days": len(window),
            }
        )

    moves = pd.DataFrame(rows)
    moves["month"] = moves["curr_expiry"].dt.month
    moves["year"] = moves["curr_expiry"].dt.year
    moves["range_pct"] = moves["max_up_pct"] - moves["max_down_pct"]
    return moves


def flag_suspect_days(price: pd.DataFrame, threshold: float = GAP_FLAG_PCT) -> pd.Series:
    """Single-session close-to-close moves large enough to suggest an unadjusted corporate action."""
    change = price["close"].pct_change() * 100
    return change[change.abs() > threshold]


price_df = load_candles(SYMBOL)
suspect = flag_suspect_days(price_df)

print(
    f"{SYMBOL.upper()}: {len(price_df):,} daily candles, "
    f"{price_df.index.min():%Y-%m-%d} to {price_df.index.max():%Y-%m-%d}"
)
print(f"{len(expiry_sessions_for(price_df))} monthly expiries inside that history")
if suspect.empty:
    print(f"No single-day move beyond ±{GAP_FLAG_PCT:.0f}% — nothing that looks like an unadjusted corporate action")
else:
    print(f"⚠ {len(suspect)} single-day move(s) beyond ±{GAP_FLAG_PCT:.0f}% — check for splits/bonus in these cycles:")
    for ts, value in suspect.items():
        print(f"   {ts:%Y-%m-%d}  {value:+.1f}%")
display(price_df.tail())

RELIANCE: 2,890 daily candles, 2015-01-01 to 2026-08-31
140 monthly expiries inside that history
No single-day move beyond ±25% — nothing that looks like an unadjusted corporate action


,open,high,low,close,volume
timestamp,,,,,
2026-08-25,"1,304.30","1,317.10","1,300.00","1,317.00",7115355
2026-08-26,"1,310.00","1,315.60","1,298.00","1,298.00",5744474
2026-08-27,"1,305.00","1,308.40","1,282.20","1,282.20",11271497
2026-08-28,"1,284.90","1,291.80","1,280.00","1,287.00",6830228
2026-08-31,"1,278.70","1,297.60","1,271.00","1,277.00",34871137


## Expiry-To-Expiry Cycles

- **base_close** — close on the previous expiry; everything below is measured against it
- **max_up_pct** — highest high in the cycle vs `base_close`
- **max_down_pct** — lowest low in the cycle vs `base_close`
- **expiry_move_pct** — the close-to-close move actually settled at the next expiry

In [4]:
moves = build_windows(price_df, SKIP_FIRST_CANDLES)
print(f"{SYMBOL.upper()}: {len(moves)} expiry-to-expiry cycles (skip_first_candles={SKIP_FIRST_CANDLES})")
display(moves.tail(10))

RELIANCE: 139 expiry-to-expiry cycles (skip_first_candles=0)


,prev_expiry,curr_expiry,base_close,max_up_pct,max_up_date,max_down_pct,max_down_date,expiry_close,expiry_move_pct,trading_days,month,year,range_pct
129,2025-10-28,2025-11-25,"1,486.90",4.89,2025-11-25,-1.13,2025-11-04,"1,539.70",3.55,19,11,2025,6.02
130,2025-11-25,2025-12-30,"1,539.70",2.70,2025-11-28,-1.44,2025-12-04,"1,539.80",0.01,24,12,2025,4.14
131,2025-12-30,2026-01-27,"1,539.80",4.68,2026-01-05,-11.16,2026-01-27,"1,380.50",-10.35,18,1,2026,15.83
132,2026-01-27,2026-02-24,"1,380.50",7.90,2026-02-03,-3.30,2026-02-01,"1,428.80",3.50,21,2,2026,11.19
133,2026-02-24,2026-03-30,"1,428.80",0.82,2026-02-25,-8.52,2026-03-04,"1,343.90",-5.94,22,3,2026,9.34
134,2026-03-30,2026-04-28,"1,343.90",3.82,2026-04-28,-4.01,2026-04-06,"1,388.90",3.35,18,4,2026,7.83
135,2026-04-28,2026-05-26,"1,388.90",6.08,2026-05-05,-5.49,2026-05-20,"1,356.30",-2.35,19,5,2026,11.58
136,2026-05-26,2026-06-30,"1,356.30",0.94,2026-05-29,-7.60,2026-06-11,"1,293.90",-4.60,23,6,2026,8.54
137,2026-06-30,2026-07-28,"1,293.90",4.02,2026-07-20,-3.41,2026-07-24,"1,267.70",-2.02,20,7,2026,7.43
138,2026-07-28,2026-08-25,"1,267.70",5.47,2026-08-07,0.08,2026-07-29,"1,317.00",3.89,20,8,2026,5.39


## Summary Stats

In [5]:
def summary_stats(moves: pd.DataFrame, symbol: str) -> pd.DataFrame:
    return pd.DataFrame(
        [
            ("Symbol", symbol.upper()),
            ("Expiry cycles analyzed", f"{len(moves)}"),
            ("First / last expiry", f"{moves['curr_expiry'].min():%Y-%m-%d} → {moves['curr_expiry'].max():%Y-%m-%d}"),
            ("Trading days per cycle (median)", f"{moves['trading_days'].median():.0f}"),
            ("Avg max up %", f"{moves['max_up_pct'].mean():.2f}%"),
            ("Avg max down %", f"{moves['max_down_pct'].mean():.2f}%"),
            ("Avg intra-cycle range (up − down)", f"{moves['range_pct'].mean():.2f}%"),
            (
                "Biggest cycle rally",
                f"{moves['max_up_pct'].max():.2f}% (expiry {moves.loc[moves['max_up_pct'].idxmax(), 'curr_expiry'].date()})",
            ),
            (
                "Biggest cycle crash",
                f"{moves['max_down_pct'].min():.2f}% (expiry {moves.loc[moves['max_down_pct'].idxmin(), 'curr_expiry'].date()})",
            ),
            ("Avg expiry-to-expiry close move", f"{moves['expiry_move_pct'].mean():.2f}%"),
            ("Median absolute settle move", f"{moves['expiry_move_pct'].abs().median():.2f}%"),
            ("Cycles closed higher", f"{(moves['expiry_move_pct'] > 0).sum()} / {len(moves)}"),
            ("Cycles closed lower", f"{(moves['expiry_move_pct'] < 0).sum()} / {len(moves)}"),
        ],
        columns=["Metric", "Value"],
    )


display(summary_stats(moves, SYMBOL))

,Metric,Value
0,Symbol,RELIANCE
1,Expiry cycles analyzed,139
2,First / last expiry,2015-02-26 → 2026-08-25
3,Trading days per cycle (median),20
4,Avg max up %,6.91%
5,Avg max down %,-4.57%
6,Avg intra-cycle range (up − down),11.48%
7,Biggest cycle rally,40.21% (expiry 2020-04-30)
8,Biggest cycle crash,-36.84% (expiry 2020-03-26)
9,Avg expiry-to-expiry close move,1.60%


## Visual Table — Max Up % / Max Down % Per Expiry Cycle

Most recent cycle first. Shading encodes magnitude — deeper warm for a larger max upside, deeper cool for a larger max downside — with the signed value printed in every cell, so direction never rests on colour alone.

In [6]:
def shade(values: pd.Series, color: tuple[int, int, int]) -> list[str]:
    """Blend from the neutral surface toward `color` in proportion to each value's magnitude."""
    magnitude = values.abs()
    peak = magnitude.max() or 1.0
    intensity = (magnitude / peak).clip(0, 1)
    return [
        "rgb({},{},{})".format(*[int(NEUTRAL[i] + (color[i] - NEUTRAL[i]) * t) for i in range(3)])
        for t in intensity
    ]


def move_table_figure(moves: pd.DataFrame, symbol: str) -> go.Figure:
    table_df = moves.sort_values("curr_expiry", ascending=False).reset_index(drop=True)
    white = ["white"] * len(table_df)
    close_colors = [
        f"rgb{UP_COLOR}" if v > 0 else (f"rgb{DOWN_COLOR}" if v < 0 else f"rgb{NEUTRAL}")
        for v in table_df["expiry_move_pct"]
    ]

    fig = go.Figure(
        data=[
            go.Table(
                columnwidth=[95, 95, 90, 85, 105, 90, 105, 95],
                header=dict(
                    values=[
                        "Prev Expiry", "Expiry", "Base Close", "Max Up %", "Max Up Date",
                        "Max Down %", "Max Down Date", "Expiry Move %",
                    ],
                    fill_color=HEADER_FILL,
                    font=dict(color="white", size=12),
                    align="center",
                    height=32,
                ),
                cells=dict(
                    values=[
                        table_df["prev_expiry"].dt.strftime("%Y-%m-%d"),
                        table_df["curr_expiry"].dt.strftime("%Y-%m-%d"),
                        table_df["base_close"].map("{:,.2f}".format),
                        table_df["max_up_pct"].map("{:+.2f}%".format),
                        table_df["max_up_date"].dt.strftime("%Y-%m-%d"),
                        table_df["max_down_pct"].map("{:+.2f}%".format),
                        table_df["max_down_date"].dt.strftime("%Y-%m-%d"),
                        table_df["expiry_move_pct"].map("{:+.2f}%".format),
                    ],
                    fill_color=[
                        white, white, white,
                        shade(table_df["max_up_pct"], UP_COLOR),
                        white,
                        shade(table_df["max_down_pct"], DOWN_COLOR),
                        white,
                        close_colors,
                    ],
                    align="center",
                    height=26,
                    font=dict(size=11),
                ),
            )
        ]
    )
    fig.update_layout(
        title=f"{symbol.upper()} — Max Up / Max Down % From Prior Monthly Expiry",
        height=900,
        margin=dict(t=50, b=10, l=10, r=10),
    )
    return fig


move_table_figure(moves, SYMBOL).show()

## Touched vs Settled — The Option Seller's Ladder

For each distance from the prior expiry close, two very different probabilities:

- **Touched** — the stock traded through that level at some point inside the cycle. This is what stops out a managed short.
- **Settled beyond** — it was still past that level at expiry. This is what actually pays out on a held short.

Stocks touch far more often than they settle beyond, which is exactly why monthly stock strangles look attractive on settlement statistics and uncomfortable on the path.

In [7]:
def ladder_frame(moves: pd.DataFrame, distances: list[float]) -> pd.DataFrame:
    ladder = pd.DataFrame({"distance_pct": distances})
    ladder["touch_up"] = [(moves["max_up_pct"] >= d).mean() * 100 for d in distances]
    ladder["touch_down"] = [(moves["max_down_pct"] <= -d).mean() * 100 for d in distances]
    ladder["settle_above"] = [(moves["expiry_move_pct"] >= d).mean() * 100 for d in distances]
    ladder["settle_below"] = [(moves["expiry_move_pct"] <= -d).mean() * 100 for d in distances]
    ladder["touched_either"] = [
        ((moves["max_up_pct"] >= d) | (moves["max_down_pct"] <= -d)).mean() * 100 for d in distances
    ]
    ladder["settled_outside"] = [(moves["expiry_move_pct"].abs() >= d).mean() * 100 for d in distances]
    ladder["strangle_survived"] = 100 - ladder["settled_outside"]
    return ladder


def ladder_table_figure(ladder: pd.DataFrame, moves: pd.DataFrame, symbol: str) -> go.Figure:
    fig = go.Figure(
        data=[
            go.Table(
                columnwidth=[80, 80, 80, 85, 85, 95, 95, 105],
                header=dict(
                    values=[
                        "Distance", "Touched<br>Up", "Touched<br>Down", "Settled<br>Above",
                        "Settled<br>Below", "Touched<br>Either Side", "Settled<br>Outside",
                        "Short Strangle<br>Expired Worthless",
                    ],
                    fill_color=HEADER_FILL,
                    font=dict(color="white", size=11),
                    align="center",
                    height=44,
                ),
                cells=dict(
                    values=[
                        ladder["distance_pct"].map("±{:.0f}%".format),
                        ladder["touch_up"].map("{:.1f}%".format),
                        ladder["touch_down"].map("{:.1f}%".format),
                        ladder["settle_above"].map("{:.1f}%".format),
                        ladder["settle_below"].map("{:.1f}%".format),
                        ladder["touched_either"].map("{:.1f}%".format),
                        ladder["settled_outside"].map("{:.1f}%".format),
                        ladder["strangle_survived"].map("{:.1f}%".format),
                    ],
                    fill_color=[
                        ["#f4f3f0"] * len(ladder),
                        shade(ladder["touch_up"], UP_COLOR),
                        shade(ladder["touch_down"], DOWN_COLOR),
                        shade(ladder["settle_above"], UP_COLOR),
                        shade(ladder["settle_below"], DOWN_COLOR),
                        ["white"] * len(ladder),
                        ["white"] * len(ladder),
                        ["white"] * len(ladder),
                    ],
                    align="center",
                    height=28,
                    font=dict(size=11),
                ),
            )
        ]
    )
    fig.update_layout(
        title=f"{symbol.upper()} — Touch vs Settle Probability By Distance ({len(moves)} cycles)",
        height=120 + 28 * len(ladder) + 60,
        margin=dict(t=60, b=10, l=10, r=10),
    )
    return fig


def ladder_bar_figure(ladder: pd.DataFrame, symbol: str) -> go.Figure:
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=ladder["distance_pct"], y=ladder["touched_either"], name="Touched either side",
            marker_color=SERIES_HEX[0], marker_line=dict(color="white", width=2),
            text=ladder["touched_either"].map("{:.0f}%".format), textposition="outside",
        )
    )
    fig.add_trace(
        go.Bar(
            x=ladder["distance_pct"], y=ladder["settled_outside"], name="Settled outside at expiry",
            marker_color=SERIES_HEX[1], marker_line=dict(color="white", width=2),
            text=ladder["settled_outside"].map("{:.0f}%".format), textposition="outside",
        )
    )
    fig.update_layout(
        title=f"{symbol.upper()} — Path Risk vs Settlement Risk By Strike Distance",
        barmode="group", bargap=0.25, template="plotly_white", height=440, hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(t=80, b=50, l=60, r=20),
    )
    fig.update_xaxes(title_text="Strike distance from prior expiry close", ticksuffix="%", tickvals=ladder["distance_pct"])
    fig.update_yaxes(title_text="% of cycles", ticksuffix="%", range=[0, 105])
    return fig


ladder = ladder_frame(moves, STRIKE_DISTANCES)
ladder_table_figure(ladder, moves, SYMBOL).show()
ladder_bar_figure(ladder, SYMBOL).show()

## Return Distribution — Expiry-To-Expiry Close Move %

Every cycle's settled move bucketed into 2.5% ranges, best to worst, with the probability of landing at or beyond each bound. Colour runs warm for gains, cool for losses, neutral through the middle.

In [8]:
def diverging(n: int) -> list[str]:
    """Warm (best) to neutral to cool (worst), matching the up/down poles used throughout."""
    colors = []
    for t in np.linspace(1.0, -1.0, n):
        pole = UP_COLOR if t >= 0 else DOWN_COLOR
        mix = abs(t)
        colors.append("rgb({},{},{})".format(*[int(NEUTRAL[i] + (pole[i] - NEUTRAL[i]) * mix) for i in range(3)]))
    return colors


def distribution_frame(moves: pd.DataFrame) -> pd.DataFrame:
    edges = np.arange(LOWER_TAIL, UPPER_TAIL + BIN_WIDTH, BIN_WIDTH)
    bin_edges = [-np.inf, *edges, np.inf]

    labels = [f"< {LOWER_TAIL:.1f}%"]
    labels += [f"{lo:.1f}% to {hi:.1f}%" for lo, hi in zip(edges[:-1], edges[1:])]
    labels.append(f"> {UPPER_TAIL:.1f}%")

    bucket = pd.cut(moves["expiry_move_pct"], bins=bin_edges, labels=labels, right=False)
    dist = bucket.value_counts().reindex(labels[::-1]).rename("periods").to_frame()   # best bucket first
    total = int(dist["periods"].sum())
    dist["pct_of_total"] = dist["periods"] / total * 100
    dist["prob_at_least"] = dist["periods"].cumsum() / total * 100
    dist["prob_at_most"] = 100 - dist["prob_at_least"] + dist["pct_of_total"]
    return dist


def distribution_figure(dist: pd.DataFrame, symbol: str) -> go.Figure:
    row_colors = diverging(len(dist))
    total = int(dist["periods"].sum())
    fig = go.Figure(
        data=[
            go.Table(
                columnwidth=[140, 110, 90, 140, 140],
                header=dict(
                    values=[
                        "Return Range", "Number of Cycles", "% of Total",
                        "Probability (≥ lower bound)", "Probability (≤ upper bound)",
                    ],
                    fill_color=HEADER_FILL, font=dict(color="white", size=12), align="center", height=34,
                ),
                cells=dict(
                    values=[
                        dist.index,
                        dist["periods"],
                        dist["pct_of_total"].map("{:.1f}%".format),
                        dist["prob_at_least"].map("{:.1f}%".format),
                        dist["prob_at_most"].map("{:.1f}%".format),
                    ],
                    fill_color=[row_colors] * 5,
                    font=dict(color="black", size=11),
                    align="center", height=28,
                ),
            )
        ]
    )
    fig.update_layout(
        title=f"{symbol.upper()} — Monthly Expiry Return Distribution ({total} cycles)",
        height=120 + 28 * len(dist) + 40,
        margin=dict(t=50, b=10, l=10, r=10),
    )
    return fig


distribution_figure(distribution_frame(moves), SYMBOL).show()

## ✈️ Survivability Analysis — Outlier Expiry Cycles

Inspired by Abraham Wald's WWII aircraft survivability analysis: the bullet holes on bombers that made it home showed where a plane could be hit and still fly — the damage that mattered was on the planes that never came back. For a premium seller the cycles that settled near the middle of the distribution are the planes that returned; the ones worth studying are the cycles that settled out in the tails.

Each cycle's settled move (`expiry_move_pct`) is ranked against the symbol's own history. The central `OUTLIER_COVERAGE`% of cycles form the survivable band; a cycle that settled below the band's lower percentile is a **downside outlier**, above its upper percentile an **upside outlier**. Drag the slider to widen or tighten the band — the summary, scatter and outlier table re-render together.

The table marks any outlier cycle containing a single-day move beyond ±`GAP_FLAG_PCT`%. For stocks that is either a genuine crash (March 2020) or an unadjusted corporate action (the ADANIENT demerger in June 2015), so check those rows before trusting the tail.

In [ ]:
from typing import NamedTuple

import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import Markdown

WITHIN_HEX = "#a3a19b"   # neutral grey — cycles that settled inside the band
OUTLIER_STYLE = {         # status → (marker colour, marker size)
    "Within band": (WITHIN_HEX, 8),
    "Downside outlier": (DOWN_HEX, 11),
    "Upside outlier": (UP_HEX, 11),
}


class OutlierSplit(NamedTuple):
    cycles: pd.DataFrame   # every cycle, oldest first, with `status` and `big_gap_day`
    coverage_pct: float
    lower: float           # band edges, as % settled move
    upper: float


def classify_outliers(moves: pd.DataFrame, coverage_pct: float, suspect: pd.Series) -> OutlierSplit:
    """Split cycles into the central `coverage_pct` band of settled moves and the tails either side."""
    tail = (100 - coverage_pct) / 2
    lower, upper = np.percentile(moves["expiry_move_pct"], [tail, 100 - tail])

    cycles = moves.sort_values("curr_expiry").reset_index(drop=True)
    cycles["status"] = np.select(
        [cycles["expiry_move_pct"] < lower, cycles["expiry_move_pct"] > upper],
        ["Downside outlier", "Upside outlier"],
        default="Within band",
    )
    cycles["big_gap_day"] = [
        bool(((suspect.index > prev) & (suspect.index <= curr)).any())
        for prev, curr in zip(cycles["prev_expiry"], cycles["curr_expiry"])
    ]
    return OutlierSplit(cycles, coverage_pct, lower, upper)


def outlier_summary(split: OutlierSplit, symbol: str) -> Markdown:
    total = len(split.cycles)
    n_down = int((split.cycles["status"] == "Downside outlier").sum())
    n_up = int((split.cycles["status"] == "Upside outlier").sum())
    tail = (100 - split.coverage_pct) / 2
    return Markdown(
        f"**{symbol.upper()}** — {total} cycles. The {split.coverage_pct:.0f}% band runs from the "
        f"**{tail:.1f}th** to the **{100 - tail:.1f}th percentile** of settled moves: "
        f"**{split.lower:+.2f}%** to **{split.upper:+.2f}%**.  \n"
        f"Outliers: **{n_down + n_up}** ({(n_down + n_up) / total * 100:.1f}%) — "
        f"**{n_down}** downside, **{n_up}** upside."
    )


def outlier_scatter_figure(split: OutlierSplit, symbol: str) -> go.Figure:
    fig = go.Figure()
    for status, (color, size) in OUTLIER_STYLE.items():
        sub = split.cycles[split.cycles["status"] == status]
        hover = sub[["prev_expiry", "max_up_pct", "max_down_pct"]].assign(
            prev_expiry=sub["prev_expiry"].dt.strftime("%Y-%m-%d")
        )
        fig.add_trace(
            go.Scatter(
                x=sub["curr_expiry"], y=sub["expiry_move_pct"], mode="markers",
                name=f"{status} ({len(sub)})",
                marker=dict(color=color, size=size, line=dict(color="white", width=2)),
                customdata=hover.to_numpy(dtype=object),
                hovertemplate=(
                    "%{customdata[0]} → %{x|%Y-%m-%d}<br>Settled %{y:+.2f}%<br>"
                    "Max up %{customdata[1]:+.2f}% · max down %{customdata[2]:+.2f}%"
                    f"<extra>{status}</extra>"
                ),
            )
        )

    fig.add_hline(y=0, line_color="#8a8880", line_width=1)
    for bound, color, label, position in [
        (split.upper, UP_HEX, "Upper", "top left"),
        (split.lower, DOWN_HEX, "Lower", "bottom left"),
    ]:
        fig.add_hline(
            y=bound, line_dash="dash", line_color=color, line_width=1.5,
            annotation_text=f"{label} boundary {bound:+.2f}%", annotation_position=position,
            annotation_font_color="#3d3c38",
        )

    fig.update_layout(
        title=f"{symbol.upper()} — Settled Move Per Cycle vs The {split.coverage_pct:.0f}% Survivability Band",
        template="plotly_white", height=500, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(t=90, b=50, l=60, r=20),
    )
    fig.update_xaxes(title_text="Expiry (end of cycle)", gridcolor="#eceae6")
    fig.update_yaxes(title_text="Settled move from prior expiry close", ticksuffix="%", gridcolor="#eceae6")
    return fig


def blend(color: tuple[int, int, int], t: float) -> str:
    """Mix `t` of `color` into the neutral surface (0 = neutral, 1 = full pole)."""
    return "rgb({},{},{})".format(*[int(NEUTRAL[i] + (color[i] - NEUTRAL[i]) * t) for i in range(3)])


def signed_shade(values: pd.Series) -> list[str]:
    """Warm for gains, cool for losses, both scaled against the largest absolute value."""
    peak = values.abs().max() or 1.0
    return [blend(UP_COLOR if v > 0 else DOWN_COLOR, min(abs(v) / peak, 1.0)) for v in values]


def outlier_table_figure(split: OutlierSplit, symbol: str) -> go.Figure:
    out = split.cycles[split.cycles["status"] != "Within band"]
    out = out.loc[out["expiry_move_pct"].abs().sort_values(ascending=False).index]
    white = ["white"] * len(out)
    is_down = out["status"] == "Downside outlier"

    fig = go.Figure(
        data=[
            go.Table(
                columnwidth=[95, 95, 100, 90, 95, 110, 120],
                header=dict(
                    values=[
                        "Prev Expiry", "Expiry", "Settled Move %", "Max Up %", "Max Down %",
                        "Status", f"Day Beyond ±{GAP_FLAG_PCT:.0f}%",
                    ],
                    fill_color=HEADER_FILL, font=dict(color="white", size=12), align="center", height=32,
                ),
                cells=dict(
                    values=[
                        out["prev_expiry"].dt.strftime("%Y-%m-%d"),
                        out["curr_expiry"].dt.strftime("%Y-%m-%d"),
                        out["expiry_move_pct"].map("{:+.2f}%".format),
                        out["max_up_pct"].map("{:+.2f}%".format),
                        out["max_down_pct"].map("{:+.2f}%".format),
                        np.where(is_down, "▼ Downside", "▲ Upside"),
                        np.where(out["big_gap_day"], "⚠ check", ""),
                    ],
                    fill_color=[
                        white, white,
                        signed_shade(out["expiry_move_pct"]),
                        shade(out["max_up_pct"], UP_COLOR),
                        shade(out["max_down_pct"], DOWN_COLOR),
                        [blend(DOWN_COLOR if down else UP_COLOR, 0.35) for down in is_down],
                        white,
                    ],
                    align="center", height=26, font=dict(size=11),
                ),
            )
        ]
    )
    fig.update_layout(
        title=f"{symbol.upper()} — {len(out)} Cycles Settled Outside The {split.coverage_pct:.0f}% Band, Largest First",
        height=min(700, 120 + 26 * max(len(out), 1)),
        margin=dict(t=50, b=10, l=10, r=10),
    )
    return fig


def render_outliers(coverage_pct=OUTLIER_COVERAGE):
    split = classify_outliers(moves, coverage_pct, suspect)
    display(outlier_summary(split, SYMBOL))
    outlier_scatter_figure(split, SYMBOL).show()
    outlier_table_figure(split, SYMBOL).show()


interact(
    render_outliers,
    coverage_pct=widgets.IntSlider(
        value=OUTLIER_COVERAGE, min=50, max=99, step=1,
        description="Probability boundary coverage (%)", continuous_update=False,
        style={"description_width": "initial"}, layout=widgets.Layout(width="500px"),
    ),
)

interactive(children=(IntSlider(value=80, continuous_update=False, description='Probability boundary coverage …

<function __main__.render_outliers(coverage_pct=80)>

## Seasonality — Average Cycle By Calendar Month

Average max upside and max downside of the cycle *ending* in each calendar month. Thin sample per month (roughly one cycle per year), so read it as texture rather than signal.

In [10]:
MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

by_month = (
    moves.groupby("month")
    .agg(cycles=("expiry_move_pct", "size"),
         avg_up=("max_up_pct", "mean"),
         avg_down=("max_down_pct", "mean"),
         avg_settle=("expiry_move_pct", "mean"))
    .reindex(range(1, 13))
)
by_month.index = MONTH_NAMES

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=by_month.index, y=by_month["avg_up"], name="Avg max up %",
        marker_color=UP_HEX, marker_line=dict(color="white", width=2),
    )
)
fig.add_trace(
    go.Bar(
        x=by_month.index, y=by_month["avg_down"], name="Avg max down %",
        marker_color=DOWN_HEX, marker_line=dict(color="white", width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=by_month.index, y=by_month["avg_settle"], name="Avg settled move %",
        mode="lines+markers", line=dict(color="#1baf7a", width=2), marker=dict(size=8),
    )
)
fig.update_layout(
    title=f"{SYMBOL.upper()} — Average Cycle Excursion By Calendar Month Of Expiry",
    barmode="relative", template="plotly_white", height=460, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(t=80, b=50, l=60, r=20),
)
fig.update_yaxes(title_text="% from prior expiry close", ticksuffix="%", gridcolor="#eceae6")
fig.add_hline(y=0, line_color="#8a8880", line_width=1)
fig.show()

display(by_month.style.format({"cycles": "{:.0f}", "avg_up": "{:+.2f}%", "avg_down": "{:+.2f}%", "avg_settle": "{:+.2f}%"}))

,cycles,avg_up,avg_down,avg_settle
Jan,11,+6.88%,-4.27%,-1.02%
Feb,12,+6.86%,-4.79%,+0.81%
Mar,12,+7.15%,-6.66%,+0.44%
Apr,12,+9.49%,-3.78%,+6.47%
May,12,+4.35%,-5.98%,-1.92%
Jun,12,+9.50%,-2.28%,+5.03%
Jul,12,+8.90%,-3.32%,+3.26%
Aug,12,+5.85%,-4.45%,+1.55%
Sep,11,+5.91%,-4.40%,+0.32%
Oct,11,+8.60%,-4.91%,+1.83%


## Cross-Symbol Screen

Every symbol in the folder, run through the same window builder, ranked by how often the cycle **settled inside** ±`SCREEN_DISTANCE`. The two columns to read together are *settled inside* (what a held short strangle keeps) and *touched either side* (how often it had to be sat through) — a stock that scores well on the first and badly on the second is a premium seller's nightmare to manage.

In [11]:
def screen_symbols(symbols: list[str], distance: float, skip_first: int = 0) -> pd.DataFrame:
    rows = []
    for symbol in symbols:
        symbol_moves = build_windows(load_candles(symbol), skip_first)
        if symbol_moves.empty:
            continue
        rows.append(
            {
                "symbol": symbol.upper(),
                "cycles": len(symbol_moves),
                "first_expiry": symbol_moves["curr_expiry"].min(),
                "avg_max_up": symbol_moves["max_up_pct"].mean(),
                "avg_max_down": symbol_moves["max_down_pct"].mean(),
                "median_abs_settle": symbol_moves["expiry_move_pct"].abs().median(),
                "p90_abs_settle": symbol_moves["expiry_move_pct"].abs().quantile(0.9),
                "settled_inside": (symbol_moves["expiry_move_pct"].abs() < distance).mean() * 100,
                "touched_either": (
                    (symbol_moves["max_up_pct"] >= distance) | (symbol_moves["max_down_pct"] <= -distance)
                ).mean() * 100,
            }
        )
    return pd.DataFrame(rows).sort_values("settled_inside", ascending=False).reset_index(drop=True)


screen = screen_symbols(SYMBOLS, SCREEN_DISTANCE, SKIP_FIRST_CANDLES)

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[95, 65, 95, 90, 95, 105, 100, 105, 110],
            header=dict(
                values=[
                    "Symbol", "Cycles", "First Expiry", "Avg Max<br>Up %", "Avg Max<br>Down %",
                    "Median Abs<br>Settle %", "P90 Abs<br>Settle %",
                    f"Settled Inside<br>±{SCREEN_DISTANCE:.0f}%", f"Touched<br>±{SCREEN_DISTANCE:.0f}%",
                ],
                fill_color=HEADER_FILL, font=dict(color="white", size=11), align="center", height=44,
            ),
            cells=dict(
                values=[
                    screen["symbol"],
                    screen["cycles"],
                    screen["first_expiry"].dt.strftime("%Y-%m-%d"),
                    screen["avg_max_up"].map("{:+.2f}%".format),
                    screen["avg_max_down"].map("{:+.2f}%".format),
                    screen["median_abs_settle"].map("{:.2f}%".format),
                    screen["p90_abs_settle"].map("{:.2f}%".format),
                    screen["settled_inside"].map("{:.1f}%".format),
                    screen["touched_either"].map("{:.1f}%".format),
                ],
                fill_color=[
                    ["#f4f3f0"] * len(screen), ["white"] * len(screen), ["white"] * len(screen),
                    shade(screen["avg_max_up"], UP_COLOR),
                    shade(screen["avg_max_down"], DOWN_COLOR),
                    ["white"] * len(screen), ["white"] * len(screen),
                    shade(screen["settled_inside"], UP_COLOR),
                    shade(screen["touched_either"], DOWN_COLOR),
                ],
                align="center", height=24, font=dict(size=11),
            ),
        )
    ]
)
fig.update_layout(
    title=f"Monthly Expiry Screen — {len(screen)} Symbols, Ranked By Cycles Settled Inside ±{SCREEN_DISTANCE:.0f}%",
    height=min(1000, 150 + 24 * len(screen)),
    margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

ranked = screen.sort_values("settled_inside")
bar = go.Figure(
    go.Bar(
        x=ranked["settled_inside"], y=ranked["symbol"], orientation="h",
        marker_color=SERIES_HEX[0], marker_line=dict(color="white", width=2),
        text=ranked["settled_inside"].map("{:.0f}%".format), textposition="outside",
        hovertemplate="%{y} — %{x:.1f}% of cycles settled inside<extra></extra>",
    )
)
bar.update_layout(
    title=f"Share Of Monthly Cycles That Settled Inside ±{SCREEN_DISTANCE:.0f}% Of The Prior Expiry Close",
    template="plotly_white", height=max(480, 20 * len(ranked) + 120),
    margin=dict(t=70, b=50, l=110, r=50),
)
bar.update_xaxes(title_text="% of cycles", ticksuffix="%", range=[0, 100], gridcolor="#eceae6")
bar.update_yaxes(title_text="")
bar.show()

## Switch Symbol Without Editing The Parameters Cell

The dropdowns re-run the per-symbol view for any stock in the folder. The coverage slider only affects the two survivability views. One view renders at a time — keeping each render to a single figure keeps the notebook's saved output small enough to execute and commit; the parameter-selected symbol still gets every section above in full.

Changing the dropdown does not change the `SYMBOL` parameter. Set that at the top (or via papermill) when you want the rest of the notebook to follow.

In [12]:
VIEWS = {
    "Touch vs settle ladder": lambda m, lad, split, s: ladder_table_figure(lad, m, s),
    "Path vs settlement bars": lambda m, lad, split, s: ladder_bar_figure(lad, s),
    "Return distribution": lambda m, lad, split, s: distribution_figure(distribution_frame(m), s),
    "Per-cycle move table": lambda m, lad, split, s: move_table_figure(m, s),
    "Survivability scatter": lambda m, lad, split, s: outlier_scatter_figure(split, s),
    "Outlier cycles table": lambda m, lad, split, s: outlier_table_figure(split, s),
}


def render_symbol(symbol=SYMBOL, view="Touch vs settle ladder", skip_first=SKIP_FIRST_CANDLES,
                  coverage_pct=OUTLIER_COVERAGE):
    symbol_price = load_candles(symbol)
    symbol_moves = build_windows(symbol_price, skip_first)
    if symbol_moves.empty:
        display(Markdown(f"**{symbol.upper()}** has no complete expiry cycle in its history."))
        return

    display(summary_stats(symbol_moves, symbol))

    suspect_days = flag_suspect_days(symbol_price)
    if not suspect_days.empty:
        flagged = ", ".join(f"{ts:%Y-%m-%d} ({value:+.1f}%)" for ts, value in suspect_days.items())
        display(Markdown(f"⚠ Possible unadjusted corporate action(s): {flagged}"))

    split = classify_outliers(symbol_moves, coverage_pct, suspect_days)
    VIEWS[view](symbol_moves, ladder_frame(symbol_moves, STRIKE_DISTANCES), split, symbol).show()


interact(
    render_symbol,
    symbol=widgets.Dropdown(options=SYMBOLS, value=SYMBOL, description="Symbol"),
    view=widgets.Dropdown(options=list(VIEWS), value="Touch vs settle ladder", description="View"),
    skip_first=widgets.IntSlider(
        value=SKIP_FIRST_CANDLES, min=0, max=5, step=1,
        description="Skip first N candles", continuous_update=False,
        style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
    ),
    coverage_pct=widgets.IntSlider(
        value=OUTLIER_COVERAGE, min=50, max=99, step=1,
        description="Survivability band coverage (%)", continuous_update=False,
        style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
    ),
)

interactive(children=(Dropdown(description='Symbol', index=35, options=('adanient', 'adaniports', 'apollohosp'…

<function __main__.render_symbol(symbol='reliance', view='Touch vs settle ladder', skip_first=0, coverage_pct=80)>